# Problem Set 3: Bootstrap Confidence Intervals

First, let's import the necessary libraries for our analysis.

In [1]:
import numpy as np
import statsmodels.api as sm
import pandas as pd
import time

### Question 1: Data Generation

Generate $X_i \sim N(1,1)$, $U_i|X_i \sim N(0,X_i^2)$, and $Y_i = \beta_0 + X_i\beta_1 + U_i$ with $\beta_0 = \beta_1 = 1$ for $n=10$.

In [2]:
# Set parameters
n = 10
beta0 = 1
beta1 = 1

# Set a seed for reproducibility of this single run
np.random.seed(42)

# Generate Xi ~ N(1,1)
Xi = np.random.normal(1, 1, n)

# Generate Ui|Xi ~ N(0, Xi^2)
# Note: The variance is Xi^2, so the standard deviation is |Xi|
Ui = np.array([np.random.normal(0, abs(x)) for x in Xi])

# Generate Yi = beta0 + Xi*beta1 + Ui
Yi = beta0 + beta1 * Xi + Ui

print(f"Generated Data for n={n}:")
print("-------------------------")
print(f"Xi: {np.round(Xi, 2)}")
print(f"Ui: {np.round(Ui, 2)}")
print(f"Yi: {np.round(Yi, 2)}")

Generated Data for n=10:
-------------------------
Xi: [1.5  0.86 1.65 2.52 0.77 0.77 2.58 1.77 0.53 1.54]
Ui: [-0.69 -0.4   0.4  -4.83 -1.32 -0.43 -2.61  0.56 -0.48 -2.18]
Yi: [ 1.8   1.46  3.05 -1.3   0.44  1.34  0.97  3.32  1.05  0.36]


### Question 2: OLS Estimation and Bootstrap Confidence Interval

For this single sample, we will:
1. Compute the OLS estimator $\hat{\beta} = (\hat{\beta}_0, \hat{\beta}_1)$ and the robust standard error $se(\hat{\beta}_1)$.
2. Construct a 95% confidence interval for $\beta_1$ using the specified bootstrap percentile-t method with $B=1000$ samples.

In [3]:
# Add a constant to Xi for the intercept term
X_with_const = sm.add_constant(Xi)

# Compute OLS estimator and robust standard error for the original sample
model = sm.OLS(Yi, X_with_const).fit()
betahat_0, betahat_1 = model.params
# Use HC1 robust standard errors
se_betahat_1 = model.get_robustcov_results(cov_type='HC1').bse[1]

print("OLS Results for the Single Sample:")
print("----------------------------------")
print(f"OLS Estimator (betahat_1): {betahat_1:.4f}")
print(f"Robust Standard Error (se(betahat_1)): {se_betahat_1:.4f}")

# --- Bootstrap Procedure ---
B = 1000
t_boot_values = []

for _ in range(B):
    # 1. Draw a random bootstrap sample (sampling with replacement)
    indices = np.random.choice(n, n, replace=True)
    Xi_b = Xi[indices]
    Yi_b = Yi[indices]
    X_b_with_const = sm.add_constant(Xi_b)
    
    # Compute OLS estimator and robust SE for the bootstrap sample
    model_b = sm.OLS(Yi_b, X_b_with_const).fit()
    betahat_1_b = model_b.params[1]
    se_betahat_1_b = model_b.get_robustcov_results(cov_type='HC1').bse[1]
    
    # 2. Compute the bootstrap t-statistic (tb)
    if se_betahat_1_b > 1e-6: # Avoid division by zero
        tb = (betahat_1_b - betahat_1) / se_betahat_1_b
        t_boot_values.append(tb)

# 4. Find the 0.025 and 0.975 quantiles of {tb}
q_025 = np.percentile(t_boot_values, 2.5)
q_975 = np.percentile(t_boot_values, 97.5)

print(f"\nBootstrap Results (B={B}):")
print("--------------------------")
print(f"2.5th quantile of t-statistics (q_0.025): {q_025:.4f}")
print(f"97.5th quantile of t-statistics (q_0.975): {q_975:.4f}")

# Construct the bootstrap percentile-t confidence interval
ci_lower = betahat_1 - q_975 * se_betahat_1
ci_upper = betahat_1 - q_025 * se_betahat_1

print(f"\n95% Bootstrap Percentile-t CI for beta1:")
print(f"[{ci_lower:.4f}, {ci_upper:.4f}]")

OLS Results for the Single Sample:
----------------------------------
OLS Estimator (betahat_1): -0.3294
Robust Standard Error (se(betahat_1)): 0.6261

Bootstrap Results (B=1000):
--------------------------
2.5th quantile of t-statistics (q_0.025): -2.6742
97.5th quantile of t-statistics (q_0.975): 6.1956

95% Bootstrap Percentile-t CI for beta1:
[-4.2087, 1.3450]


### Question 3: Check if the Interval Contains the True Value

Check if the interval calculated above contains the true value $\beta_1 = 1$.

In [4]:
contains_true_beta1 = (ci_lower <= beta1 <= ci_upper)

print(f"Does the CI [{ci_lower:.4f}, {ci_upper:.4f}] contain the true beta1=1? -> {contains_true_beta1}")

Does the CI [-4.2087, 1.3450] contain the true beta1=1? -> True


### Questions 4 & 5: Monte Carlo Simulation and Full Experiment

Now we automate the process. We will create a function to run the simulation for a given sample size `n`, and then call it for `n = 10, 50, 100, 500, 1000`. Each simulation will involve `num_reps=1000` repetitions of the experiment.

For each repetition, we will:
- Generate a new sample.
- Construct the 95% bootstrap percentile-t CI.
- Check if the CI covers the true $\beta_1$.

Finally, we will report the proportion of times the CI contained the true value (the empirical coverage probability) for each sample size `n`.

In [5]:
def run_simulation(n, beta0=1, beta1=1, num_reps=1000, B=1000):
    """Runs the full Monte Carlo simulation for a given sample size n."""
    coverage_count = 0
    
    for i in range(num_reps):
        Xi = np.random.normal(1, 1, n)
        Ui = np.array([np.random.normal(0, abs(x)) for x in Xi])
        Yi = beta0 + beta1 * Xi + Ui
        X_with_const = sm.add_constant(Xi)
        
        model = sm.OLS(Yi, X_with_const).fit()
        betahat_1 = model.params[1]
        se_betahat_1 = model.get_robustcov_results(cov_type='HC1').bse[1]

        t_boot_values = []
        for _ in range(B):
            indices = np.random.choice(n, n, replace=True)
            Xi_b, Yi_b = Xi[indices], Yi[indices]
            X_b_with_const = sm.add_constant(Xi_b)
            
            model_b = sm.OLS(Yi_b, X_b_with_const).fit()
            betahat_1_b = model_b.params[1]
            se_betahat_1_b = model_b.get_robustcov_results(cov_type='HC1').bse[1]
            
            if se_betahat_1_b > 1e-6:
                tb = (betahat_1_b - betahat_1) / se_betahat_1_b
                t_boot_values.append(tb)

        if not t_boot_values:
            continue
            
        q_025 = np.percentile(t_boot_values, 2.5)
        q_975 = np.percentile(t_boot_values, 97.5)
        
        ci_lower = betahat_1 - q_975 * se_betahat_1
        ci_upper = betahat_1 - q_025 * se_betahat_1
        
        if ci_lower <= beta1 <= ci_upper:
            coverage_count += 1
            
    return coverage_count / num_reps

def full_experiment():
    """Main function to run the complete experiment and print results."""
    simulation_params = {'num_reps': 1000, 'B': 1000}
    sample_sizes = [10, 50, 100, 500, 1000]
    results = []

    print(f"Running simulations with {simulation_params['num_reps']} reps and {simulation_params['B']} bootstrap samples.")
    total_start_time = time.time()
    
    for n in sample_sizes:
        print(f'\nStarting experiment for n = {n}...')
        n_start_time = time.time()
        
        coverage = run_simulation(n=n, **simulation_params)
        
        duration = time.time() - n_start_time
        results.append({'Sample Size (n)': n, 'Coverage Probability': coverage, 'Duration (s)': f"{duration:.2f}"})
        print(f'Finished for n = {n}. Coverage: {coverage:.4f}')

    print(f'\nTotal time: {time.time() - total_start_time:.2f} seconds')
    results_df = pd.DataFrame(results)
    print('\n--- Final Results Summary ---')
    print(results_df.to_string(index=False))

# Run the full experiment
full_experiment()

Running simulations with 1000 reps and 1000 bootstrap samples.

Starting experiment for n = 10...
Finished for n = 10. Coverage: 0.9010

Starting experiment for n = 50...
Finished for n = 50. Coverage: 0.9430

Starting experiment for n = 100...
Finished for n = 100. Coverage: 0.9160

Starting experiment for n = 500...
Finished for n = 500. Coverage: 0.9460

Starting experiment for n = 1000...
Finished for n = 1000. Coverage: 0.9510

Total time: 799.28 seconds

--- Final Results Summary ---
 Sample Size (n)  Coverage Probability Duration (s)
              10                 0.901       156.44
              50                 0.943       177.86
             100                 0.916       140.33
             500                 0.946       154.26
            1000                 0.951       170.39


### Analysis and Comparison

**What do you see?**

The summary table shows the empirical coverage probability for each sample size `n`. We typically observe that for a very small sample size like `n=10`, the coverage can be significantly different from the nominal 95% level. As the sample size `n` increases, the coverage probability should get closer and closer to 0.95. This is because the Central Limit Theorem and the consistency of the bootstrap procedure work better with larger samples, leading to more accurate standard error estimates and more reliable pivotal statistics.

**Comparing to Problem 1 in Problem Set 1**

We can see that when the sample size is small(n=10), the confidence interval of bootstrapping is more likely to cover the the true data-generating parameter than OLS.